In [ ]:
import os
import json
from dotenv import load_dotenv
from typing import List, Optional

import mlflow
from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_google_vertexai import ChatVertexAI
from langchain_community.tools.tavily_search import TavilySearchResults
import yfinance as yf



In [ ]:

# --- 1. DEFINE TOOLS ---

# Tool for fetching the stock code
def get_stock_code(company_name: str) -> str:
    """
    Fetches the primary stock ticker symbol for a given company name using yfinance.
    This function is wrapped in a RunnableLambda to be part of the LangChain pipeline.
    """
    try:
        # yfinance Ticker can be finicky with full names, let's try the name as is first.
        ticker = yf.Ticker(company_name)
        if ticker.info and ticker.info.get('symbol'):
            # Sometimes yfinance auto-corrects to the symbol, e.g., "Apple Inc" -> Ticker for AAPL
            return ticker.info['symbol']
        
        # As a fallback, let's search. This is often less reliable.
        # A more robust solution might use an LLM call or a dedicated symbol search API.
        # For this example, we'll keep it simple. If direct lookup fails, we return a message.
        return f"Could not definitively find ticker for {company_name}. Searching news by name."
    except Exception as e:
        print(f"Error fetching ticker for {company_name}: {e}")
        return f"Error fetching ticker for {company_name}."

# Tool for fetching news from Tavily
# This tool is great for getting recent, relevant article snippets.
news_search_tool = TavilySearchResults(max_results=7)


# --- 2. DEFINE THE DESIRED OUTPUT STRUCTURE (JSON Schema) ---

class SentimentProfile(BaseModel):
    """The structured output for the sentiment analysis of a company."""
    company_name: str = Field(description="The name of the company being analyzed.")
    stock_code: str = Field(description="The stock ticker symbol for the company.")
    sentiment: str = Field(description="The overall sentiment from the news. Must be one of: 'Positive', 'Negative', 'Neutral'.")
    news_summary: str = Field(description="A concise one-paragraph summary of the key news points.")
    people_names: List[str] = Field(description="List of prominent people's names mentioned in the articles.")
    places_names: List[str] = Field(description="List of relevant cities, states, or countries mentioned.")
    other_companies_referred: List[str] = Field(description="List of other company names mentioned in the news.")
    related_industries: List[str] = Field(description="List of industries related to the news content (e.g., 'Artificial Intelligence', 'Automotive').")
    market_implications: str = Field(description="A brief analysis of the potential market implications based on the news.")
    confidence_score: float = Field(description="A confidence score (0.0 to 1.0) for the sentiment classification.", ge=0.0, le=1.0)


# --- 3. INITIALIZE LLM and OUTPUT PARSER ---

# Initialize the Google Gemini model via Vertex AI
llm = ChatVertexAI(
    model="gemini-2.0-flash", # Using the specified fast model
    temperature=0.0,
    project=os.getenv("541947596059"),
    location=os.getenv("India")
)

# Initialize the Pydantic parser to enforce the JSON schema
output_parser = PydanticOutputParser(pydantic_object=SentimentProfile)


# --- 4. CREATE THE PROMPT TEMPLATE ---

prompt_template = ChatPromptTemplate.from_template(
    """
    You are an expert financial analyst. Your task is to analyze the provided news articles about a company
    and generate a structured sentiment profile in JSON format.

    Analyze the news content below for the company: "{company_name}".

    **News Articles:**
    {news_articles}

    **Instructions:**
    1.  Read all articles carefully to understand the overall sentiment and key information.
    2.  Determine the overall sentiment (Positive, Negative, or Neutral) based on the balance of the news.
    3.  Extract all relevant named entities as requested in the format instructions.
    4.  Provide a concise summary and market implications based *only* on the provided text.
    5.  Fill out all fields in the JSON format as instructed.

    **Output Format Instructions:**
    {format_instructions}
    """
)


# --- 5. BUILD THE LANGCHAIN PIPELINE (CHAIN) ---

# The chain is built using LangChain Expression Language (LCEL) for clarity and composability.

# Step A: Define the data gathering pipeline. This runs in parallel.
# It accepts a dictionary with a "company" key (e.g., {"company": "Apple Inc."})
gather_data_chain = RunnableParallel(
    # Fetches the stock code using our custom function
    stock_code=RunnableLambda(lambda x: get_stock_code(x["company"])),
    # Fetches news articles using the Tavily tool
    news_articles=lambda x: news_search_tool.invoke(f"Latest financial news about {x['company']}"),
    # Passes the original company name through for use in the final prompt
    company_name=lambda x: x["company"],
    # Passes the format instructions from the parser through
    format_instructions=lambda x: output_parser.get_format_instructions(),
)

# Step B: Define the full analysis pipeline
# 1. `gather_data_chain`: First, fetch the stock code and news in parallel.
# 2. `prompt_template`: The collected data is then fed into the prompt template.
# 3. `llm`: The formatted prompt is sent to the Gemini LLM.
# 4. `output_parser`: The LLM's string output (which should be JSON) is parsed into the Pydantic object.
sentiment_analyzer_chain = gather_data_chain | prompt_template | llm | output_parser




In [ ]:
# --- 6. EXECUTE THE CHAIN ---
if __name__ == "__main__":
    print("Starting Company Sentiment Analysis Pipeline...")

    company_to_analyze = "Google"

    # Using an MLflow run context to group all logs for this execution
    with mlflow.start_run() as run:
        mlflow.log_param("company_name", company_to_analyze)
        print(f"Analyzing company: {company_to_analyze}")

        # The .invoke() method executes the chain.
        # LangChain + MLflow autologging will trace every step.
        result = sentiment_analyzer_chain.invoke({"company": company_to_analyze})

        # Pretty-print the final Pydantic object as a JSON string
        output_json = result.model_dump_json(indent=2)
        print("\nAnalysis Complete. Result:")
        print(output_json)

        # Log the final JSON output as an artifact in MLflow
        mlflow.log_text(output_json, "sentiment_profile.json")
        print(f"\nMLflow Run ID: {run.info.run_id}")
        print(f"Run 'mlflow ui' in your terminal and navigate to the experiment to see the trace.")